In [0]:
%skip
%sql
CREATE SCHEMA IF NOT EXISTS hvacapp_dev3.raw;

CREATE OR REPLACE TABLE hvacapp_dev3.raw.bronze_hvac_telemetry (
  event_id STRING NOT NULL,
  event_ts TIMESTAMP NOT NULL,
  ingest_ts TIMESTAMP NOT NULL,
  site_id STRING NOT NULL,
  building_id STRING NOT NULL,
  equipment_id STRING NOT NULL,
  sensor_id STRING NOT NULL,
  metric_name STRING NOT NULL,
  metric_value DOUBLE NOT NULL,
  unit STRING NOT NULL,
  quality_flag STRING,
  source_system STRING,
  raw_payload STRING
)
USING DELTA;

SELECT * FROM hvacapp_dev3.raw.bronze_hvac_telemetry;

In [0]:
%skip
sensor_df = spark.table("hvacapp_dev3.master.sensor")
display(sensor_df)

In [0]:
%skip
# very simple first simulator

# reads sensor master table
# loops through each sensor
# creates one fake reading for each sensor
# puts the results into a Spark DataFrame

# At this point, it does not write yet. It only shows the generated rows (5 rows).

from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType
from datetime import datetime
import uuid
import random

# Read sensor master data
sensors = spark.table("hvacapp_dev3.master.sensor").collect()

# Current simulation timestamp
current_ts = datetime.now()

rows = []

for s in sensors:
    metric_name = s["metric_name"]
    sensor_id = s["sensor_id"]
    equipment_id = s["equipment_id"]
    site_id = s["site_id"]
    building_id = s["building_id"]
    unit = s["unit"]

    # Generate fake values based on sensor metric
    if metric_name == "chw_supply_temp_c":
        value = round(random.uniform(7.5, 9.5), 2)
    elif metric_name == "chw_return_temp_c":
        value = round(random.uniform(10.5, 13.5), 2)
    elif metric_name == "hvac_power_kw":
        value = round(random.uniform(300.0, 420.0), 2)
    elif metric_name == "chw_flow_lpm":
        value = round(random.uniform(650.0, 900.0), 2)
    elif metric_name == "pump_speed_pct":
        value = round(random.uniform(65.0, 85.0), 2)
    else:
        value = 0.0

    rows.append(
        Row(
            event_id=str(uuid.uuid4()),
            event_ts=current_ts,
            ingest_ts=current_ts,
            site_id=site_id,
            building_id=building_id,
            equipment_id=equipment_id,
            sensor_id=sensor_id,
            metric_name=metric_name,
            metric_value=value,
            unit=unit,
            quality_flag="GOOD",
            source_system="SIMULATOR",
            raw_payload=None
        )
    )



In [0]:
%skip
# Create DataFrame
schema = StructType([StructField('event_id', StringType(), True), StructField('event_ts', TimestampType(), True), StructField('ingest_ts', TimestampType(), True), StructField('site_id', StringType(), True), StructField('building_id', StringType(), True), StructField('equipment_id', StringType(), True), StructField('sensor_id', StringType(), True), StructField('metric_name', StringType(), True), StructField('metric_value', DoubleType(), True), StructField('unit', StringType(), True), StructField('quality_flag', StringType(), True), StructField('source_system', StringType(), True), StructField('raw_payload', StringType(), True)])

sim_df = spark.createDataFrame(rows, schema=schema)
display(sim_df)

In [0]:
%skip
# Write simulator output into the bronze table

sim_df.write.mode("append").saveAsTable("hvacapp_dev3.raw.bronze_hvac_telemetry")

In [0]:
%skip
%sql
-- # Validate the inserted data

SELECT *
FROM hvacapp_dev3.raw.bronze_hvac_telemetry
ORDER BY event_ts DESC;

In [0]:
%skip
# improved version simulator
# This version is better because:

# return temperature is linked to supply temperature
# power is influenced by flow and temperature difference
# the whole system looks more realistic

from pyspark.sql import Row
from datetime import datetime
import uuid
import random

sensors = spark.table("hvacapp_dev3.master.sensor").collect()

current_ts = datetime.now()

# Create one operating state first
base_supply_temp = round(random.uniform(7.8, 9.2), 2)
base_return_temp = round(base_supply_temp + random.uniform(2.5, 4.5), 2)
base_flow_lpm = round(random.uniform(700.0, 850.0), 2)
base_pump_speed = round(random.uniform(68.0, 82.0), 2)

# Simple power logic
base_power_kw = round(
    180 + (base_flow_lpm * 0.12) + ((base_return_temp - base_supply_temp) * 18) + random.uniform(-10, 10),
    2
)

rows = []

for s in sensors:
    metric_name = s["metric_name"]
    sensor_id = s["sensor_id"]
    equipment_id = s["equipment_id"]
    site_id = s["site_id"]
    building_id = s["building_id"]
    unit = s["unit"]

    if metric_name == "chw_supply_temp_c":
        value = base_supply_temp
    elif metric_name == "chw_return_temp_c":
        value = base_return_temp
    elif metric_name == "hvac_power_kw":
        value = base_power_kw
    elif metric_name == "chw_flow_lpm":
        value = base_flow_lpm
    elif metric_name == "pump_speed_pct":
        value = base_pump_speed
    else:
        value = 0.0

    rows.append(
        Row(
            event_id=str(uuid.uuid4()),
            event_ts=current_ts,
            ingest_ts=current_ts,
            site_id=site_id,
            building_id=building_id,
            equipment_id=equipment_id,
            sensor_id=sensor_id,
            metric_name=metric_name,
            metric_value=float(value),
            unit=unit,
            quality_flag="GOOD",
            source_system="SIMULATOR",
            raw_payload=None
        )
    )





In [0]:
%skip
# Create DataFrame
schema = StructType([StructField('event_id', StringType(), True), StructField('event_ts', TimestampType(), True), StructField('ingest_ts', TimestampType(), True), StructField('site_id', StringType(), True), StructField('building_id', StringType(), True), StructField('equipment_id', StringType(), True), StructField('sensor_id', StringType(), True), StructField('metric_name', StringType(), True), StructField('metric_value', DoubleType(), True), StructField('unit', StringType(), True), StructField('quality_flag', StringType(), True), StructField('source_system', StringType(), True), StructField('raw_payload', StringType(), True)])

sim_df = spark.createDataFrame(rows, schema=schema)

display(sim_df)

In [0]:
%skip
# Write again with improved version
sim_df.write.mode("append").saveAsTable("hvacapp_dev3.raw.bronze_hvac_telemetry")

In [0]:
%skip
%sql
SELECT COUNT(*) AS total_rows
FROM hvacapp_dev3.raw.bronze_hvac_telemetry;

-- help confirm that all metrics are being generated.
SELECT metric_name, COUNT(*) AS row_count
FROM hvacapp_dev3.raw.bronze_hvac_telemetry
GROUP BY metric_name
ORDER BY metric_name;

In [0]:
%skip
%sql
-- Create a helper query to inspect latest readings
SELECT
  event_ts,
  equipment_id,
  sensor_id,
  metric_name,
  metric_value,
  unit
FROM hvacapp_dev3.raw.bronze_hvac_telemetry
ORDER BY event_ts DESC, metric_name;


In [0]:
# Create loops to generate many batches
# 20 batches
# 2 seconds between batches
# appends all rows to the table

from pyspark.sql import Row
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType
import uuid
import random
import time

sensors = spark.table("hvacapp_dev3.master.sensor").collect()

for i in range(50):
    current_ts = datetime.now()

    base_supply_temp = round(random.uniform(7.8, 9.2), 2)
    base_return_temp = round(base_supply_temp + random.uniform(2.5, 4.5), 2)
    base_flow_lpm = round(random.uniform(700.0, 850.0), 2)
    base_pump_speed = round(random.uniform(68.0, 82.0), 2)

    base_power_kw = round(
        180 + (base_flow_lpm * 0.12) + ((base_return_temp - base_supply_temp) * 18) + random.uniform(-10, 10),
        2
    )

    rows = []

    for s in sensors:
        metric_name = s["metric_name"]
        sensor_id = s["sensor_id"]
        equipment_id = s["equipment_id"]
        site_id = s["site_id"]
        building_id = s["building_id"]
        unit = s["unit"]

        if metric_name == "chw_supply_temp_c":
            value = base_supply_temp
        elif metric_name == "chw_return_temp_c":
            value = base_return_temp
        elif metric_name == "hvac_power_kw":
            value = base_power_kw
        elif metric_name == "chw_flow_lpm":
            value = base_flow_lpm
        elif metric_name == "pump_speed_pct":
            value = base_pump_speed
        else:
            value = 0.0

        rows.append(
            Row(
                event_id=str(uuid.uuid4()),
                event_ts=current_ts,
                ingest_ts=current_ts,
                site_id=site_id,
                building_id=building_id,
                equipment_id=equipment_id,
                sensor_id=sensor_id,
                metric_name=metric_name,
                metric_value=float(value),
                unit=unit,
                quality_flag="GOOD",
                source_system="SIMULATOR",
                raw_payload=None
            )
        )

# Create DataFrame
schema = StructType([
    StructField('event_id', StringType(), True), 
    StructField('event_ts', TimestampType(), True), 
    StructField('ingest_ts', TimestampType(), True), 
    StructField('site_id', StringType(), True), 
    StructField('building_id', StringType(), True), 
    StructField('equipment_id', StringType(), True), 
    StructField('sensor_id', StringType(), True), 
    StructField('metric_name', StringType(), True), 
    StructField('metric_value', DoubleType(), True), 
    StructField('unit', StringType(), True), 
    StructField('quality_flag', StringType(), True), 
    StructField('source_system', StringType(), True), 
    StructField('raw_payload', StringType(), True)])

batch_df = spark.createDataFrame(rows, schema=schema)
batch_df.write.mode("append").saveAsTable("hvacapp_dev3.raw.bronze_hvac_telemetry")

print(f"Batch {i+1} inserted at {current_ts}")
time.sleep(2)

In [0]:
%skip
%sql
-- Check the result after loop

SELECT
  date_trunc('minute', event_ts) AS event_minute,
  metric_name,
  AVG(metric_value) AS avg_value
FROM hvacapp_dev3.raw.bronze_hvac_telemetry
GROUP BY date_trunc('minute', event_ts), metric_name
ORDER BY event_minute DESC, metric_name;

-- DELETE FROM hvacapp_dev3.raw.bronze_hvac_telemetry
